In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA student_gold;
DESCRIBE TABLE workspace.student_silver.student_exams_clean;

col_name,data_type,comment
source_file,string,null
loaded_at,timestamp,null
student_id,string,null
age,int,null
gender,string,null
education_level,string,null
school_type,string,null
family_income,string,null
parent_education,string,null
urban_rural,string,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.student_gold.dim_student
USING DELTA
AS

SELECT
    SHA2(student_id, 256) AS student_key,
    student_id,
    age,
    age_group,
    gender,
    education_level,
    school_type,
    family_income,
    parent_education,
    urban_rural

FROM workspace.student_silver.student_exams_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.student_gold.dim_study_profile
USING DELTA
AS

SELECT DISTINCT

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'class_participation', class_participation,
                'private_tuition', private_tuition,
                'study_consistency', study_consistency,
                'study_environment', study_environment,
                'study_method', study_method,
                'revision_frequency', revision_frequency,
                'notes_quality', notes_quality
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS study_profile_key,

    class_participation,
    private_tuition,
    study_consistency,
    study_environment,
    study_method,
    revision_frequency,
    notes_quality

FROM workspace.student_silver.student_exams_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.student_gold.dim_lifestyle_profile
USING DELTA
AS

SELECT DISTINCT

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'sleep_quality', sleep_quality,
                'break_frequency', break_frequency,
                'motivation_level', motivation_level
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS lifestyle_key,

    sleep_quality,
    break_frequency,
    motivation_level

FROM workspace.student_silver.student_exams_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.student_gold.dim_digital_access
USING DELTA
AS

SELECT DISTINCT

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'internet_access', internet_access,
                'device_availability', device_availability,
                'educational_app_usage', educational_app_usage
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS digital_key,

    internet_access,
    device_availability,
    educational_app_usage

FROM workspace.student_silver.student_exams_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.student_gold.fact_exam_performance
USING DELTA
AS

SELECT

    SHA2(student_id, 256) AS exam_performance_key,

    -- Foreign keys

    SHA2(student_id, 256) AS student_key,

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'class_participation', class_participation,
                'private_tuition', private_tuition,
                'study_consistency', study_consistency,
                'study_environment', study_environment,
                'study_method', study_method,
                'revision_frequency', revision_frequency,
                'notes_quality', notes_quality
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS study_profile_key,

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'sleep_quality', sleep_quality,
                'break_frequency', break_frequency,
                'motivation_level', motivation_level
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS lifestyle_key,

    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'internet_access', internet_access,
                'device_availability', device_availability,
                'educational_app_usage', educational_app_usage
            ),
            MAP('ignoreNullFields', 'false')
        ),
        256
    ) AS digital_key,

    -- Previous academic performance

    previous_exam_score,
    previous_gpa,
    attendance_percentage,
    assignment_completion_rate,

    -- Study habits

    study_hours_per_day,
    self_study_hours,
    online_learning_hours,
    practice_tests_completed,

    -- Lifestyle

    sleep_hours,
    daily_screen_time,
    physical_activity_hours,
    stress_level,

    -- Digital learning

    online_course_hours,

    -- Exam preparation

    exam_difficulty,
    exam_preparation_days,
    time_management_score,
    exam_anxiety_level,

    -- Exam performance

    questions_attempted,
    questions_correct,
    exam_score,
    performance_grade,
    pass_status,
    performance_level,

    -- Derived analytical features

    study_hours_group,
    sleep_hours_group,
    attendance_group,
    screen_time_group,
    exam_score_change,
    question_accuracy_pct,
    is_passed,

    -- Technical metadata

    CURRENT_TIMESTAMP() AS gold_processed_at

FROM workspace.student_silver.student_exams_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SHOW TABLES IN workspace.student_gold;

SELECT
    'dim_student' AS table_name,
    COUNT(*) AS row_count
FROM workspace.student_gold.dim_student

UNION ALL

SELECT
    'dim_study_profile',
    COUNT(*)
FROM workspace.student_gold.dim_study_profile

UNION ALL

SELECT
    'dim_lifestyle_profile',
    COUNT(*)
FROM workspace.student_gold.dim_lifestyle_profile

UNION ALL

SELECT
    'dim_digital_access',
    COUNT(*)
FROM workspace.student_gold.dim_digital_access

UNION ALL

SELECT
    'fact_exam_performance',
    COUNT(*)
FROM workspace.student_gold.fact_exam_performance;

table_name,row_count
dim_student,100000
dim_study_profile,3194
dim_lifestyle_profile,45
dim_digital_access,18
fact_exam_performance,100000


In [0]:
%sql
--sprawdzenie unikalności kluczy głównych

SELECT
    'dim_student' AS table_name,
    COUNT(*) - COUNT(DISTINCT student_key)
        AS invalid_keys
FROM workspace.student_gold.dim_student

UNION ALL

SELECT
    'dim_study_profile',
    COUNT(*) - COUNT(DISTINCT study_profile_key)
FROM workspace.student_gold.dim_study_profile

UNION ALL

SELECT
    'dim_lifestyle_profile',
    COUNT(*) - COUNT(DISTINCT lifestyle_key)
FROM workspace.student_gold.dim_lifestyle_profile

UNION ALL

SELECT
    'dim_digital_access',
    COUNT(*) - COUNT(DISTINCT digital_key)
FROM workspace.student_gold.dim_digital_access

UNION ALL

SELECT
    'fact_exam_performance',
    COUNT(*) - COUNT(DISTINCT exam_performance_key)
FROM workspace.student_gold.fact_exam_performance;

table_name,invalid_keys
dim_student,0
dim_study_profile,0
dim_lifestyle_profile,0
dim_digital_access,0
fact_exam_performance,0


In [0]:
%sql
-- sprawdzenie integralności referencyjnej

SELECT

    COUNT(*) AS joined_rows,

    COUNT_IF(s.student_key IS NULL)
        AS missing_student_keys,

    COUNT_IF(st.study_profile_key IS NULL)
        AS missing_study_keys,

    COUNT_IF(l.lifestyle_key IS NULL)
        AS missing_lifestyle_keys,

    COUNT_IF(d.digital_key IS NULL)
        AS missing_digital_keys

FROM workspace.student_gold.fact_exam_performance f

LEFT JOIN workspace.student_gold.dim_student s
    ON f.student_key = s.student_key

LEFT JOIN workspace.student_gold.dim_study_profile st
    ON f.study_profile_key = st.study_profile_key

LEFT JOIN workspace.student_gold.dim_lifestyle_profile l
    ON f.lifestyle_key = l.lifestyle_key

LEFT JOIN workspace.student_gold.dim_digital_access d
    ON f.digital_key = d.digital_key;

joined_rows,missing_student_keys,missing_study_keys,missing_lifestyle_keys,missing_digital_keys
100000,0,0,0,0


In [0]:
%sql

-- test

SELECT

    st.study_method,

    COUNT(*) AS student_count,

    ROUND(AVG(f.exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(f.study_hours_per_day), 2)
        AS avg_study_hours,

    ROUND(
        AVG(CAST(f.is_passed AS INT)) * 100,
        2
    ) AS pass_rate_pct

FROM workspace.student_gold.fact_exam_performance f

INNER JOIN workspace.student_gold.dim_study_profile st
    ON f.study_profile_key = st.study_profile_key

GROUP BY st.study_method

ORDER BY avg_exam_score DESC;

study_method,student_count,avg_exam_score,avg_study_hours,pass_rate_pct
Self-Reading,19805,62.11,3.0,77.91
Summarizing,20048,62.0,3.0,77.6
Practice Tests,19891,61.99,2.99,77.51
Flashcards,20239,61.83,2.98,76.99
Group Study,20017,61.82,3.01,76.94
